#### Step 3: Subsample to token budget

**Why:** Step 4's full dataset run measured 18.41 billion tokens total (English 5.85B, Hindi 9.71B, Marathi 2.85B). We only want a 10 billion token training set, and instead of just proportionally shrinking everything (which keeps today's language mix), we want a fixed target mix: **Hindi 40%, English 35%, Marathi 25%**.

**How we decide what to drop:** within each language we have two sources: `wikipedia` (small, curated, high quality) and `sangraha` or `fineweb-edu` (the bulk, web/crawl based). For each language we keep the small curated source fully, and only trim the big source down to whatever is left in that language's token budget. This avoids ever throwing away any Wikipedia data, and concentrates the drop on the largest, most redundant source.

This notebook reads the exact per shard token counts from Step 4 (`../4_tokenization/full_dataset_token_counts.csv`), works out a keep ratio per (language, source), then randomly samples that fraction of rows from every shard in `dataset/standardized/` and writes the kept rows to `dataset/sampled_10b/<language>/<source>/`. It does not touch the original standardized shards.

In [1]:
from pathlib import Path

import pandas as pd

DATASET_ROOT = Path("../../../../dataset").resolve()
STANDARDIZED_ROOT = DATASET_ROOT / "standardized"
OUTPUT_ROOT = DATASET_ROOT / "sampled_10b"
TOKEN_COUNTS_CSV = Path("../4_tokenization/full_dataset_token_counts.csv")

TARGET_TOTAL_TOKENS = 10_000_000_000
TARGET_LANGUAGE_RATIOS = {"hi": 0.40, "en": 0.35, "mr": 0.25}
SEED = 42

assert abs(sum(TARGET_LANGUAGE_RATIOS.values()) - 1.0) < 1e-9, "ratios must add up to 1.0"

token_counts = pd.read_csv(TOKEN_COUNTS_CSV)
by_source = token_counts.groupby(["language", "source"])[["rows", "tokens"]].sum()
by_source

rows      tokens
language source                           
en       fineweb-edu   5131000  5152744529
         wikipedia      781445   699546914
hi       sangraha     17420932  9645452161
         wikipedia      163093    67603894
mr       sangraha      5865617  2822044705
         wikipedia       94133    24954015

#### Work out a keep ratio per (language, source)

For each language: sort its sources smallest first, keep a source fully if it fits inside the remaining budget, otherwise keep whatever fraction of it still fits. Since Wikipedia is always far smaller than Sangraha/FineWeb-Edu here, this naturally keeps every Wikipedia row and only trims the big source.

In [2]:
keep_ratio = {}

for language, target_share in TARGET_LANGUAGE_RATIOS.items():
    target_tokens = TARGET_TOTAL_TOKENS * target_share
    remaining = target_tokens

    sources = by_source.loc[language].sort_values("tokens")
    for source, row in sources.iterrows():
        source_tokens = row["tokens"]
        if source_tokens <= remaining:
            keep_ratio[(language, source)] = 1.0
            remaining -= source_tokens
        else:
            keep_ratio[(language, source)] = remaining / source_tokens if source_tokens else 0.0
            remaining = 0.0

plan_rows = []
for (language, source), ratio in keep_ratio.items():
    total_rows = by_source.loc[(language, source), "rows"]
    total_tokens = by_source.loc[(language, source), "tokens"]
    plan_rows.append({
        "language": language,
        "source": source,
        "keep_ratio": ratio,
        "rows_total": total_rows,
        "rows_keep": round(total_rows * ratio),
        "rows_drop": round(total_rows * (1 - ratio)),
        "tokens_keep_est": round(total_tokens * ratio),
    })

plan = pd.DataFrame(plan_rows).sort_values(["language", "source"])
print(f"estimated kept tokens total: {plan['tokens_keep_est'].sum():,}  (target: {TARGET_TOTAL_TOKENS:,})")
plan

estimated kept tokens total: 10,000,000,000  (target: 10,000,000,000)


,language,source,keep_ratio,rows_total,rows_keep,rows_drop,tokens_keep_est
3,en,fineweb-edu,0.543488,5131000,2788635,2342365,2800453086
2,en,wikipedia,1.000000,781445,781445,0,699546914
1,hi,sangraha,0.407694,17420932,7102415,10318517,3932396106
0,hi,wikipedia,1.000000,163093,163093,0,67603894
5,mr,sangraha,0.877040,5865617,5144381,721236,2475045985
4,mr,wikipedia,1.000000,94133,94133,0,24954015


#### Apply the sampling

For every shard, take a random `keep_ratio` fraction of its rows (seeded, so this is reproducible) and write it to `dataset/sampled_10b/<language>/<source>/`. `rows_kept`/`rows_dropped` below are the real counts from this run. The token totals here are still estimates (based on Step 4's per shard averages) since we are not re running the tokenizer, run `4_tokenization/tokenizer_eval.ipynb` against `dataset/sampled_10b/` afterward to get the exact real token count.

In [3]:
from tqdm.auto import tqdm

shards = sorted(STANDARDIZED_ROOT.glob("*/*/*.parquet"))
print(f"found {len(shards)} shard(s) to sample from")

sample_summary = []
shard_bar = tqdm(shards, desc="shards", unit="shard")
for shard in shard_bar:
    language = shard.parent.parent.name
    source = shard.parent.name
    shard_bar.set_postfix(current=f"{language}/{source}/{shard.name}")

    ratio = keep_ratio.get((language, source), 1.0)
    df = pd.read_parquet(shard)

    if ratio >= 1.0:
        kept = df
    else:
        kept = df.sample(frac=ratio, random_state=SEED)

    out_dir = OUTPUT_ROOT / language / source
    out_dir.mkdir(parents=True, exist_ok=True)
    kept.to_parquet(out_dir / shard.name, index=False)

    sample_summary.append({
        "shard": f"{language}/{source}/{shard.name}",
        "keep_ratio": ratio,
        "rows_total": len(df),
        "rows_kept": len(kept),
        "rows_dropped": len(df) - len(kept),
    })

sample_summary = pd.DataFrame(sample_summary)
sample_summary.to_csv(OUTPUT_ROOT / "sampling_summary.csv", index=False)
print(f"\ntotal rows kept: {sample_summary['rows_kept'].sum():,}")
print(f"total rows dropped: {sample_summary['rows_dropped'].sum():,}")

/home/contributor/users/prashant.takale/github_v2/indic_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


found 149 shard(s) to sample from


shards: 100%|██████████| 149/149 [05:47<00:00,  2.33s/shard, current=mr/wikipedia/train-00000-of-00001.parquet]


total rows kept: 16,074,090
total rows dropped: 13,382,130
